# Simulated Greenhouse Plant (MODEL-01)

Open-loop 24h run of the coupled energy + moisture ODE plant (`src/greenhouse_model.py`) driven by the seed-7 weather CSV. Actuators: vent 25% always, heater 80% at night (h<6 or h>=20), fogger 50% 12:00-15:00.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.greenhouse_model import step, load_weather_profile, daytime_solar, params
from src.weather_profile_generator import generate_weather_profile

print("params:", params)

In [ ]:
# Weather input (D-10): regenerate the seed-7 profile only if absent, then load.
import os
CSV = "weather/weather_profile.csv"
if not os.path.exists(CSV):
    generate_weather_profile(seed=7, output_path=CSV)

t_min, T_out, w_out = load_weather_profile(CSV)
hours = np.arange(0, 1440) / 60.0          # dt = 60 s -> one step per minute
T_out_hr = np.interp(hours, t_min, T_out)
w_out_hr = np.interp(hours, t_min, w_out)

# Open-loop actuator schedule: vent 25% always, heater 80% at night,
# fogger 50% from 12:00-15:00.
def schedule(h):
    heater = 0.8 if (h < 6.0 or h >= 20.0) else 0.0
    fogger = 0.5 if (12.0 <= h < 15.0) else 0.0
    return {"heater": heater, "fan_cooler": 0.0, "fogger": fogger, "vent": 0.25}

In [ ]:
T, w = 29.28, 20.0                  # initial condition (probe-verified baseline)
T_trace, w_trace, rh_trace = [], [], []
for i, h in enumerate(hours):
    ws = (T_out_hr[i], w_out_hr[i], daytime_solar(h, params["A_sol"]))
    T, w, rh = step(60.0, schedule(h), ws, T, w)
    T_trace.append(T); w_trace.append(w); rh_trace.append(rh)
print("T [%.2f, %.2f]  w [%.2f, %.2f]  RH [%.1f, %.1f]" %
      (min(T_trace), max(T_trace), min(w_trace), max(w_trace), min(rh_trace), max(rh_trace)))

In [ ]:
# Three stacked traces: inside vs outdoor T, absolute humidity, RH.
plt.figure(figsize=(10, 9))
plt.subplot(3, 1, 1)
plt.plot(hours, T_trace, label="inside T")
plt.plot(hours, T_out_hr, label="outdoor T", linestyle="--")
plt.ylabel("Temperature (degC)"); plt.legend(); plt.grid(alpha=0.3)

plt.subplot(3, 1, 2)
plt.plot(hours, w_trace, color="C2")
plt.plot(hours, w_out_hr, color="C3", linestyle="--")
plt.ylabel("Abs. humidity (g/kg)"); plt.grid(alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(hours, rh_trace, color="C4")
plt.xlabel("hour of day"); plt.ylabel("RH (%)"); plt.grid(alpha=0.3)

plt.suptitle("Simulated greenhouse plant - 24h open-loop (vent 25%, heater 80% night, fogger 50% noon)")
plt.tight_layout()
plt.savefig("notebooks/01_greenhouse_model.png", dpi=120)
plt.show()